<a href="https://colab.research.google.com/github/uin33273/GoogleColab_practice/blob/main/%E7%AE%97%E5%AE%9A%E5%9F%BA%E6%BA%963%E5%80%A4%E3%82%92%E3%82%B3%E3%83%94%E3%83%BC%E3%81%99%E3%82%8B%E3%81%9F%E3%82%81%E3%81%AE%E3%83%87%E3%83%BC%E3%82%BF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

算定基準をspreadsheetへコピペするためのファイル作成

3算定基準

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from google.colab import files
import io

# --- 1. 最初のファイル（集計表）の読み込み ---
print("1.「算定区分・加算等集計表.xlsx」を選択してください。")
uploaded_agg = files.upload()
agg_key = list(uploaded_agg.keys())[0]

wb_agg = openpyxl.load_workbook(io.BytesIO(uploaded_agg[agg_key]), data_only=True)
if "集計" not in wb_agg.sheetnames:
    print("エラー：『集計』シートが見つかりません。")
else:
    ws_agg = wb_agg["集計"]
    shop_names = []
    for r in range(3, ws_agg.max_row + 1):
        val = ws_agg.cell(row=r, column=4).value
        if val is None or str(val).strip() == "":
            break
        shop_names.append(val)
    print(f"店舗名を {len(shop_names)} 件取得しました。")
    wb_agg.close()

# --- 2. 2番目のファイル（前工程で作ったファイル）の読み込み ---
print("\n2.「2-spreadsheetと複数から１つへ.csv」を選択してください。")
uploaded_target = files.upload()
target_key = list(uploaded_target.keys())[0]

if target_key.lower().endswith('.csv'):
    try:
        df = pd.read_csv(io.BytesIO(uploaded_target[target_key]), encoding='cp932')
    except:
        df = pd.read_csv(io.BytesIO(uploaded_target[target_key]), encoding='shift_jis')

    output = io.BytesIO()
    with pd.ExcelWriter(output, engine='openpyxl') as writer:
        df.to_excel(writer, index=False)
    output.seek(0)
    wb_target = openpyxl.load_workbook(output)
else:
    wb_target = openpyxl.load_workbook(io.BytesIO(uploaded_target[target_key]))

# --- 3. シート操作とデータ貼り付け ---
ws_mismatch = wb_target.active
ws_mismatch.title = "店舗名紐づけ"

if "LOOKUP" in wb_target.sheetnames:
    del wb_target["LOOKUP"]
ws_lookup = wb_target.create_sheet("LOOKUP")

for i, name in enumerate(shop_names, start=1):
    ws_lookup.cell(row=i, column=1).value = name

for col in range(3, 12):
    source_val = ws_mismatch.cell(row=1, column=col).value
    ws_lookup.cell(row=1, column=col-1).value = source_val

row_count = len(shop_names)
search_max_row = 404

# --- 4. XLOOKUP数式の入力（#N/Aを0にする修正） ---
for r in range(2, row_count + 2): # shop_namesの数に合わせて+1
    for c in range(2, 11):
        col_letter = get_column_letter(c + 1)
        # XLOOKUPの第4引数に 0 を追加: XLOOKUP(検索値, 検索範囲, 戻り範囲, [見つからない場合])
        formula = f'=_xlfn.XLOOKUP($A{r}, 店舗名紐づけ!$K$2:$K${search_max_row}, 店舗名紐づけ!{col_letter}$2:{col_letter}${search_max_row}, 0)'
        ws_lookup.cell(row=r, column=c).value = formula

# --- 5. 保存とダウンロード ---
final_name = "3-spreadsheetへコピペ用データ.xlsx"
wb_target.save(final_name)
print(f"\n完了しました。ファイルをダウンロードします: {final_name}")
files.download(final_name)

1.「算定区分・加算等集計表.xlsx」を選択してください。


Saving 算定区分・加算等集計表（運営用）.xlsx to 算定区分・加算等集計表（運営用）.xlsx
店舗名を 231 件取得しました。

2.「2-spreadsheetと複数から１つへ.csv」を選択してください。


Saving 2-spreadsheetと複数から１つへ.csv to 2-spreadsheetと複数から１つへ.csv

完了しました。ファイルをダウンロードします: 3-spreadsheetへコピペ用データ.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>